# Pre-processing Parking Violations

Dieses Notebook lädt die NYC Parking Violations Rohdaten der Fiskaljahre 2023, 2024 und 2025 aus dem HDFS, vereinheitlicht die Datenstruktur, bereinigt zentrale Felder und speichert die verarbeiteten Daten als Parquet-Dateien im HDFS.

## Ziel

- Rohdaten aus HDFS laden
- Fiscal Year ergänzen
- zentrale Spalten auswählen
- Spaltennamen vereinheitlichen
- Textfelder bereinigen
- `Issue Date` parsen
- zusätzliche Datumsfelder ableiten
- `Violation Time` bereinigen
- `violation_hour` und `violation_minute` ableiten
- fehlende Werte behandeln
- bereinigte Daten als partitioniertes Parquet speichern

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, year, month, dayofweek,
    trim, upper, coalesce, regexp_extract, when
)
from pyspark.sql import functions as F
import re

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Preprocessing") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "4") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 17:49:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/23 17:49:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/23 17:49:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
!hdfs dfs -ls -R /parking_violations/raw

drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:53 /parking_violations/raw/2023
-rw-r--r--   1 cluster supergroup 4025724719 2026-05-07 21:53 /parking_violations/raw/2023/parking_violations_2023.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:08 /parking_violations/raw/2024
-rw-r--r--   1 cluster supergroup 3002584102 2026-05-07 21:08 /parking_violations/raw/2024/parking_violations_2024.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 20:40 /parking_violations/raw/2025
-rw-r--r--   1 cluster supergroup 3064758292 2026-05-07 20:40 /parking_violations/raw/2025/parking_violations_2025.csv


In [3]:
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

raw_paths, processed_path

({2023: 'hdfs:///parking_violations/raw/2023/parking_violations_2023.csv',
  2024: 'hdfs:///parking_violations/raw/2024/parking_violations_2024.csv',
  2025: 'hdfs:///parking_violations/raw/2025/parking_violations_2025.csv'},
 'hdfs:///parking_violations/processed/parking_violations_cleaned')

In [4]:
dfs = []

for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(
        path,
        header=True,
        inferSchema=False
    ).withColumn("Fiscal Year", lit(fiscal_year))
    
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

df_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

26/05/23 17:49:56 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/23 17:50:03 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 17:50:18 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 17:50:33 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 17:50:48 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 17:51:03 WARN Task

KeyboardInterrupt: 

In [ ]:
selected_columns = [
    "Fiscal Year",
    "Summons Number",
    "Plate ID",
    "Registration State",
    "Issue Date",
    "Violation Time",
    "Violation County",
    "Violation Precinct",
    "Street Name",
    "Vehicle Make",
    "Vehicle Body Type",
    "Violation Code",
    "Violation Description"
]

existing_columns = [c for c in selected_columns if c in df_raw.columns]

df_selected = df_raw.select(existing_columns)

df_selected.show(10, truncate=False)

In [ ]:
def normalize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")
    return name

df_clean_names = df_selected.toDF(*[normalize_column_name(c) for c in df_selected.columns])

df_clean_names.columns

In [ ]:
df_clean = df_clean_names \
    .withColumn("registration_state", upper(trim(col("registration_state")))) \
    .withColumn("vehicle_make", upper(trim(col("vehicle_make")))) \
    .withColumn("vehicle_body_type", upper(trim(col("vehicle_body_type")))) \
    .withColumn("violation_county", upper(trim(col("violation_county")))) \
    .withColumn("street_name", trim(col("street_name"))) \
    .withColumn("violation_description", coalesce(col("violation_description"), lit("Unknown"))) \
    .withColumn("vehicle_make", coalesce(col("vehicle_make"), lit("Unknown"))) \
    .withColumn("vehicle_body_type", coalesce(col("vehicle_body_type"), lit("Unknown"))) \
    .withColumn("violation_county", coalesce(col("violation_county"), lit("Unknown")))

df_clean.show(10, truncate=False)

In [ ]:
df_clean = df_clean.withColumn(
    "issue_date_parsed",
    to_date(col("issue_date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
).withColumn(
    "issue_month",
    month(col("issue_date_parsed"))
).withColumn(
    "issue_weekday",
    dayofweek(col("issue_date_parsed"))
)

df_clean.select(
    "fiscal_year",
    "issue_date",
    "issue_date_parsed",
    "issue_year",
    "issue_month",
    "issue_weekday"
).show(20, truncate=False)

In [ ]:
from pyspark.sql.functions import regexp_extract, when

df_clean = df_clean.withColumn(
    "violation_time_clean",
    upper(trim(col("violation_time")))
).withColumn(
    "time_hour_raw",
    regexp_extract(col("violation_time_clean"), r"^(\d{1,2})\d{2}[AP]$", 1).cast("int")
).withColumn(
    "violation_minute",
    regexp_extract(col("violation_time_clean"), r"^\d{1,2}(\d{2})[AP]$", 1).cast("int")
).withColumn(
    "time_ampm",
    regexp_extract(col("violation_time_clean"), r"^[0-9]{3,4}([AP])$", 1)
)

df_clean = df_clean.withColumn(
    "violation_minute",
    when(
        (col("violation_minute") >= 0) & (col("violation_minute") <= 59),
        col("violation_minute")
    )
).withColumn(
    "violation_hour",
    when(
        (col("time_ampm") == "A") & (col("time_hour_raw") == 12),
        0
    ).when(
        (col("time_ampm") == "A") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw")
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw") == 12),
        12
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw") + 12
    )
)

In [ ]:
df_clean.select(
    "violation_time",
    "violation_time_clean",
    "time_hour_raw",
    "violation_minute",
    "time_ampm",
    "violation_hour"
).show(30, truncate=False)

In [ ]:
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("violation_time").isNull().cast("int")).alias("missing_original_violation_time"),
    spark_sum(col("violation_hour").isNull().cast("int")).alias("missing_parsed_violation_hour")
).show()

In [ ]:
df_clean.filter(
    col("violation_time").isNotNull() & col("violation_hour").isNull()
).select(
    "violation_time",
    "violation_time_clean"
).distinct().show(50, truncate=False)

In [ ]:
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("issue_date_parsed").isNull().cast("int")).alias("missing_issue_date_parsed")
).show()

In [ ]:
import pandas as pd
from pyspark.sql.functions import broadcast

mapping_file = "../../data_sample/ParkingViolationCodes_January2020.xlsx"

violation_code_mapping_pd = pd.read_excel(mapping_file)

violation_code_mapping_pd = violation_code_mapping_pd.rename(columns={
    "VIOLATION CODE": "violation_code",
    "VIOLATION DESCRIPTION": "violation_description_official",
    "Manhattan  96th St. & below\n(Fine Amount $)": "fine_manhattan_96_below",
    "All Other Areas\n(Fine Amount $)": "fine_other_areas"
})

violation_code_mapping_pd["violation_code"] = violation_code_mapping_pd["violation_code"].astype(str)
violation_code_mapping_pd["violation_description_official"] = violation_code_mapping_pd["violation_description_official"].astype(str)

violation_code_mapping = spark.createDataFrame(violation_code_mapping_pd)

violation_code_mapping.show(10, truncate=False)

In [ ]:
df_clean = df_clean.join(
    broadcast(violation_code_mapping),
    on="violation_code",
    how="left"
)

df_clean.select(
    "violation_code",
    "violation_description",
    "violation_description_official",
    "fine_manhattan_96_below",
    "fine_other_areas"
).show(20, truncate=False)

In [ ]:
df_clean.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

In [ ]:
df_clean_filtered = df_clean \
    .filter(col("summons_number").isNotNull()) \
    .filter(col("violation_code").isNotNull()) \
    .filter(col("issue_date_parsed").isNotNull())

df_clean_filtered.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

In [ ]:
df_clean_filtered.write.mode("overwrite") \
    .partitionBy("fiscal_year") \
    .parquet(processed_path)

In [ ]:
df_processed = spark.read.parquet(processed_path)

df_processed.printSchema()

df_processed.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

In [ ]:
from pyspark.sql.functions import sum as spark_sum

important_columns = [
    "summons_number",
    "plate_id",
    "registration_state",
    "issue_date_parsed",
    "issue_month",
    "issue_weekday",
    "violation_time",
    "violation_hour",
    "violation_minute",
    "violation_code",
    "violation_description",
    "violation_description_official",
    "fine_manhattan_96_below",
    "fine_other_areas",
    "vehicle_make",
    "vehicle_body_type",
    "violation_county",
    "fiscal_year"
]

null_check = df_processed.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
])

null_check.show(truncate=False)

In [ ]:
df_processed.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

In [ ]:
!hdfs dfs -ls /parking_violations/processed/parking_violations_cleaned

## Ergebnis

Das Pre-processing wurde erfolgreich durchgeführt. Die bereinigten Daten wurden als Parquet-Dateien im HDFS gespeichert:

`hdfs:///parking_violations/processed/parking_violations_cleaned`

Die Daten sind nach `fiscal_year` partitioniert. Dadurch können spätere Analysen pro Fiskaljahr effizienter ausgeführt werden.

Nach dem Cleaning enthält der Datensatz:

- FY2023: 21'563'238 Zeilen
- FY2024: 16'099'641 Zeilen
- FY2025: 16'557'773 Zeilen

Im Pre-processing wurden:
- zentrale Spalten ausgewählt
- Spaltennamen vereinheitlicht
- Textfelder wie `vehicle_make`, `vehicle_body_type` und `violation_county` bereinigt
- fehlende Werte in ausgewählten Textfeldern mit `Unknown` ersetzt
- `issue_date` in ein Datumsfeld umgewandelt
- `issue_year`, `issue_month` und `issue_weekday` abgeleitet
- `violation_time` bereinigt
- `violation_hour` und `violation_minute` abgeleitet
- ungültige oder uneindeutige Zeitwerte bewusst als `NULL` belassen
- die bereinigten Daten als partitioniertes Parquet gespeichert

Entfernt wurden nur Zeilen mit fehlender `summons_number`, fehlendem `violation_code` oder nicht parsebarem `issue_date`.

Die finalen Checks zeigen:
- wichtige Analysefelder wie `summons_number`, `violation_code`, `issue_date_parsed`, `issue_month`, `issue_weekday`, `vehicle_make`, `vehicle_body_type`, `violation_description`, `violation_county` und `fiscal_year` enthalten keine fehlenden Werte
- `violation_hour` enthält nur gültige Werte von `0` bis `23` oder `NULL`
- der HDFS-Output enthält `_SUCCESS` sowie Partitionen für `fiscal_year=2023`, `fiscal_year=2024` und `fiscal_year=2025`

Für spätere Tageszeit-Analysen sollen Datensätze mit `violation_hour IS NOT NULL` verwendet werden.

In [ ]:
spark.stop()

26/05/23 18:10:48 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 18:11:03 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 18:11:18 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 18:11:33 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 18:11:48 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/23 18:12:03 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th